In [ ]:
# === 1. SETUP AND IMPORTS (with new additions) ===
print("Setting up libraries...")
import pandas as pd
import numpy as np
from scipy.sparse import load_npz, hstack
from scipy.optimize import minimize  # <-- NEW: For weight optimization
from sklearn.model_selection import KFold
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import warnings

warnings.filterwarnings('ignore')
print("Setup complete.")


# === 2. DEFINE THE SMAPE METRIC ===
# We need a function to calculate SMAPE to find the optimal weights
def smape(y_true, y_pred):
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(numerator / (denominator + 1e-8)) * 100

# === 3. LOAD ALL DATA AND FEATURES ===

print("Loading all data and features...")
train_df = pd.read_csv("/kaggle/input/processed-data/train_processed.csv")
test_df = pd.read_csv("/kaggle/input/processed-data/test_processed.csv")
train_text_emb = np.load("/kaggle/input/embedded-data/train_text_embeddings.npy")
test_text_emb = np.load("/kaggle/input/embedded-data/test_text_embeddings.npy")
train_image_emb = np.load("/kaggle/input/product-image-embeddings-efficientnet/train_image_embeddings.npy")
test_image_emb = np.load("/kaggle/input/product-image-embeddings-efficientnet/test_image_embeddings.npy")
train_tfidf = load_npz("/kaggle/input/embedded-data/train_tfidf.npz")
test_tfidf = load_npz("/kaggle/input/embedded-data/test_tfidf.npz")
print("All data loaded successfully.")

# === 4. PREPARE DATA FOR MODELS ===
print("Preparing data for modeling...")
tabular_features = ['item_quantity']
X_train_tab = train_df[tabular_features].values
X_test_tab = test_df[tabular_features].values
X_train_lgbm = hstack([train_tfidf, X_train_tab])
X_test_lgbm = hstack([test_tfidf, X_test_tab])

# --- IMPORTANT: We need the original scale price for SMAPE calculation ---
# The log-transformed price is y_train_log
y_train_log = train_df['price'].values
# We also need the original price to calculate SMAPE on our validation predictions
# We apply expm1 to the log-transformed price to get it back
y_train_orig = np.expm1(y_train_log)


# Convert sparse matrices to CSR format for efficient slicing
X_train_lgbm = X_train_lgbm.tocsr()
X_test_lgbm = X_test_lgbm.tocsr()
print("Data preparation complete.")

# === 5. DEFINE THE MULTIMODAL NEURAL NETWORK (No changes here) ===
def create_multimodal_nn():
    tab_input = layers.Input(shape=(X_train_tab.shape[1],), name='tabular_input')
    text_input = layers.Input(shape=(train_text_emb.shape[1],), name='text_input')
    image_input = layers.Input(shape=(train_image_emb.shape[1],), name='image_input')
    text_branch = layers.Dense(256, activation='relu')(text_input)
    image_branch = layers.Dense(256, activation='relu')(image_input)
    concatenated = layers.concatenate([tab_input, text_branch, image_branch])
    x = layers.Dense(512, activation='relu')(concatenated)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation='linear')(x)
    model = models.Model(inputs=[tab_input, text_input, image_input], outputs=output)
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# === 6. TRAIN BASE MODELS (Simplified Ensemble) ===
print("\nStarting model training with 5-Fold CV...")
NFOLDS = 5
kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=42)

# Placeholders for OOF and test predictions
oof_lgbm = np.zeros(len(train_df))
oof_nn = np.zeros(len(train_df))
preds_lgbm = np.zeros(len(test_df))
preds_nn = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_lgbm)):
    print(f"===== FOLD {fold+1} =====")
    
    # --- LightGBM Model (with tuned parameters) ---
    print("Training LightGBM on GPU...")
    lgbm_params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'device': 'cpu',
    'n_estimators': 2000,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,      # Sample 80% of features
    'bagging_fraction': 0.8,      # Sample 80% of data
    'bagging_freq': 1,
    'num_leaves': 31,
    'max_bin': 255,               # Higher bins for better accuracy
    'min_data_in_leaf': 20,       # Prevent overfitting
    'lambda_l1': 0.1,             # L1 regularization
    'lambda_l2': 0.1,             # L2 regularization
    'verbose': -1,
    'n_jobs': -1,                 # Use all CPU cores
    'seed': 42,
    'boosting_type': 'gbdt'
}
    lgbm = lgb.LGBMRegressor(**lgbm_params)
    lgbm.fit(X_train_lgbm[train_idx], y_train_log[train_idx],
             eval_set=[(X_train_lgbm[val_idx], y_train_log[val_idx])],
             eval_metric='mae',
             callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgbm[val_idx] = lgbm.predict(X_train_lgbm[val_idx])
    preds_lgbm += lgbm.predict(X_test_lgbm) / NFOLDS

    # --- Neural Network Model ---
    print("Training Neural Network...")
    nn_model = create_multimodal_nn()
    early_stopping = callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss')
    nn_model.fit(
        [X_train_tab[train_idx], train_text_emb[train_idx], train_image_emb[train_idx]], y_train_log[train_idx],
        validation_data=([X_train_tab[val_idx], train_text_emb[val_idx], train_image_emb[val_idx]], y_train_log[val_idx]),
        epochs=50, batch_size=128, callbacks=[early_stopping], verbose=0
    )
    oof_nn[val_idx] = nn_model.predict([X_train_tab[val_idx], train_text_emb[val_idx], train_image_emb[val_idx]]).flatten()
    preds_nn += nn_model.predict([X_test_tab, test_text_emb, test_image_emb]).flatten() / NFOLDS

print("\nBase model training complete.")

# === 7. OPTIMIZE BLENDING WEIGHTS ===
print("Optimizing blending weights...")

# The OOF predictions are in log scale, so we inverse transform them
oof_lgbm_orig = np.expm1(oof_lgbm)
oof_nn_orig = np.expm1(oof_nn)

# This function takes weights and returns the SMAPE score of the blended predictions
def objective_func(weights):
    w1, w2 = weights
    # The blended prediction is a weighted average of the two models' predictions
    blended_preds = w1 * oof_lgbm_orig + w2 * oof_nn_orig
    return smape(y_train_orig, blended_preds)

# We start with equal weights and set constraints so weights sum to 1
initial_weights = [0.5, 0.5]
constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
bounds = [(0, 1) for _ in range(len(initial_weights))]

# Run the optimization
result = minimize(objective_func, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)

# Get the best weights
optimal_weights = result.x

# --- FIX: Correctly access the first element of the weights array ---
#print(f"Optimal Weights found: LGBM = {optimal_weights:.4f}, NN = {optimal_weights[1]:.4f}")
#print(f"Best OOF SMAPE score: {result.fun:.4f}")
print(f"Optimal Weights found: LGBM = {optimal_weights[0]:.4f}, NN = {optimal_weights[1]:.4f}")

# === 8. GENERATE FINAL PREDICTIONS ===
print("Generating final predictions with optimal weights...")

# The test predictions are also in log scale, so we inverse transform them
preds_lgbm_orig = np.expm1(preds_lgbm)
preds_nn_orig = np.expm1(preds_nn)

# --- FIX: Correctly apply the first weight to the first model's predictions ---
final_predictions = optimal_weights[0] * preds_lgbm_orig + optimal_weights[1] * preds_nn_orig

# Ensure all predictions are positive
final_predictions[final_predictions < 0] = 0
print("Final predictions generated.")

# === 9. CREATE SUBMISSION FILE ===
print("Creating submission file...")
submission_df = pd.DataFrame({'sample_id': test_df['sample_id'], 'price': final_predictions})
submission_df.to_csv('submission.csv', index=False)
print("\nProcess complete! 'submission.csv' has been created.")

Setting up libraries...
Setup complete.
Loading all data and features...
All data loaded successfully.
Preparing data for modeling...
Data preparation complete.

Starting model training with 5-Fold Stacking...
===== FOLD 1 =====
Training LightGBM on GPU...
Training XGBoost on GPU...
Training Ridge...
Training Neural Network...
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
===== FOLD 2 =====
Training LightGBM on GPU...
Training XGBoost on GPU...
Training Ridge...
Training Neural Network...
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
===== FOLD 3 =====
Training LightGBM on GPU...
Training XGBoost on GPU...
Training Ridge...
Training Neural Network...
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
===== FOLD 4 =====
Training LightGBM on GPU...
Training XGBoost on GPU...
Training Ridge...
Training Neural Network...
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
2344/2344 ━━━━━━━━━━━